## 1. Import delle Librerie

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

## 2. Caricamento del Dataset

In [ ]:
df = pd.read_csv('../data/data-final.csv', sep='\t')

# Anteprima delle prime 5 righe del dataset
df.head()

,EXT1,EXT2,EXT3,EXT4,EXT5,EXT6,EXT7,EXT8,EXT9,EXT10,...,dateload,screenw,screenh,introelapse,testelapse,endelapse,IPC,country,lat_appx_lots_of_err,long_appx_lots_of_err
0,4.0,1.0,5.0,2.0,5.0,1.0,5.0,2.0,4.0,1.0,...,2016-03-03 02:01:01,768.0,1024.0,9.0,234.0,6,1,GB,51.5448,0.1991
1,3.0,5.0,3.0,4.0,3.0,3.0,2.0,5.0,1.0,5.0,...,2016-03-03 02:01:20,1360.0,768.0,12.0,179.0,11,1,MY,3.1698,101.706
2,2.0,3.0,4.0,4.0,3.0,2.0,1.0,3.0,2.0,5.0,...,2016-03-03 02:01:56,1366.0,768.0,3.0,186.0,7,1,GB,54.9119,-1.3833
3,2.0,2.0,2.0,3.0,4.0,2.0,2.0,4.0,1.0,4.0,...,2016-03-03 02:02:02,1920.0,1200.0,186.0,219.0,7,1,GB,51.75,-1.25
4,3.0,3.0,3.0,3.0,5.0,3.0,3.0,5.0,3.0,4.0,...,2016-03-03 02:02:57,1366.0,768.0,8.0,315.0,17,2,KE,1.0,38.0


## 3. Pulizia dei Dati

Prima di analizzare i dati, è necessario:
- Rimuovere risposte multiple (filtro su `IPC`)
- Rimuovere risposte con valori fuori scala (le domande accettano solo valori 1–5)
- Invertire le domande formulate in negativo
- Calcolare i punteggi medi per ciascuna delle 5 dimensioni

In [3]:
# Filtro su IPC 
df = df[df['IPC'] == 1]

# Filtro sui valori validi (1–5)
domande = [col for col in df.columns
           if col[:3] in ('EXT', 'EST', 'AGR', 'CSN', 'OPN')
           and not col.endswith('_E')]

# Filtro solo le righe dove TUTTE le risposte sono tra 1 e 5
mask_valida = pd.Series(True, index=df.index)  # partiamo con tutto True

for col in domande:
    mask_valida = mask_valida & (df[col] >= 1) & (df[col] <= 5)

df = df[mask_valida]

# Filtro sui tempi di completamento del sondaggio
# Rimuovo chi ci ha messo meno di due minuti e chi più di 30.
df['tempo_completamento_min'] = df['testelapse'] / 60
df = df[(df['tempo_completamento_min'] >= 2) & (df['tempo_completamento_min'] <= 30)]


# Domande inverse
domande_inverse = [
    # Estroversione
    'EXT2', 'EXT4', 'EXT6', 'EXT8', 'EXT10',
    # Stabilità emotiva
    'EST1', 'EST3', 'EST5', 'EST6', 'EST7', 'EST8', 'EST9', 'EST10',
    # Amabilità
    'AGR1', 'AGR3', 'AGR5', 'AGR7',
    # Coscienziosità
    'CSN2', 'CSN4', 'CSN6', 'CSN8',
    # Apertura mentale
    'OPN2', 'OPN4', 'OPN6',
]

df[domande_inverse] = 6 - df[domande_inverse]

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/2692615962.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tempo_completamento_min'] = df['testelapse'] / 60


In [4]:
# Calcolo la media per ogni tratto
#  Estroversione
df['Estroversione'] = (df['EXT1'] + df['EXT2'] + df['EXT3'] + df['EXT4'] + df['EXT5'] +
                       df['EXT6'] + df['EXT7'] + df['EXT8'] + df['EXT9'] + df['EXT10']) / 10

# Stabilità emotiva
df['Stabilità_emotiva'] = (df['EST1'] + df['EST2'] + df['EST3'] + df['EST4'] + df['EST5'] +
                            df['EST6'] + df['EST7'] + df['EST8'] + df['EST9'] + df['EST10']) / 10

# Amabilità
df['Amabilità'] = (df['AGR1'] + df['AGR2'] + df['AGR3'] + df['AGR4'] + df['AGR5'] +
                   df['AGR6'] + df['AGR7'] + df['AGR8'] + df['AGR9'] + df['AGR10']) / 10

# Coscienziosità
df['Coscienziosità'] = (df['CSN1'] + df['CSN2'] + df['CSN3'] + df['CSN4'] + df['CSN5'] +
                        df['CSN6'] + df['CSN7'] + df['CSN8'] + df['CSN9'] + df['CSN10']) / 10

# Apertura mentale
df['Apertura_mentale'] = (df['OPN1'] + df['OPN2'] + df['OPN3'] + df['OPN4'] + df['OPN5'] +
                           df['OPN6'] + df['OPN7'] + df['OPN8'] + df['OPN9'] + df['OPN10']) / 10

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/3549723834.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Estroversione'] = (df['EXT1'] + df['EXT2'] + df['EXT3'] + df['EXT4'] + df['EXT5'] +
/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/3549723834.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Stabilità_emotiva'] = (df['EST1'] + df['EST2'] + df['EST3'] + df['EST4'] + df['EST5'] +
/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/3549723834.py:11: PerformanceWarn

## 4. Distribuzione dei tratti dominanti

Per ogni persona identifichiamo il **tratto dominante**, cioè quello con il punteggio medio più alto tra i cinque.
Il grafico mostra come si distribuisce questo tratto nel campione.

Apertura mentale (OPN) e Amabilità (AGR) sono di gran lunga i tratti più comuni,
coprendo insieme quasi il 73% del campione. Questo è probabilmente legato a un bias
di selezione: le persone curiose e aperte tendono ad essere più attratte da un test
di personalità online, e le persone amichevoli sono più propense a condividerlo.

In [5]:
# Per ogni persona trovo il tratto con il punteggio più alto.
df['tratto_dominante'] = df[['Estroversione', 'Stabilità_emotiva', 'Amabilità',
                              'Coscienziosità', 'Apertura_mentale']].idxmax(axis=1)

# Conto quante persone hanno ciascun tratto come dominante
conteggio = df['tratto_dominante'].value_counts().reset_index()
conteggio.columns = ['Tratto', 'Conteggio']

sigle = {
    'Estroversione':     'EXT',
    'Stabilità_emotiva': 'EST',
    'Amabilità':         'AGR',
    'Coscienziosità':    'CSN',
    'Apertura_mentale':  'OPN'
}
conteggio['Sigla'] = conteggio['Tratto'].map(sigle)

fig = px.pie(
    conteggio,
    names='Tratto',
    values='Conteggio',
    hole=0.4,
    title=f'Distribuzione dei tratti dominanti',
    custom_data=['Sigla']
)

# Sigla, percentuale e conteggio sulle fette
fig.update_traces(
    texttemplate='%{customdata[0]}<br>%{percent:.1%}',
    textposition='inside'
)

fig.show()

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/2198567921.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['tratto_dominante'] = df[['Estroversione', 'Stabilità_emotiva', 'Amabilità',


## 5. Tempo di completamento del sondaggio per tratto dominante

Confrontiamo il tempo medio impiegato per completare l'intero sondaggio
tra le persone con tratti dominanti diversi.

L'idea è verificare se certi tratti di personalità influenzano il modo
in cui si risponde al test. Per esempio, una persona con alta Coscienziosità
potrebbe prendersi più tempo per riflettere su ogni risposta, mentre una
persona molto estroversa potrebbe rispondere più d'istinto.

In [6]:

# Converto testelapse in minuti
df['tempo_completamento_min'] = df['testelapse'] / 60

# Calcolo del tempo medio per tratto dominante
tempo_per_tratto = df.groupby('tratto_dominante')['tempo_completamento_min'].mean().reset_index()
tempo_per_tratto.columns = ['Tratto', 'Tempo medio (min)']

# Ordine crescente
tempo_per_tratto = tempo_per_tratto.sort_values('Tempo medio (min)')

# Bar chart
fig = px.bar(
    tempo_per_tratto,
    x='Tratto',
    y='Tempo medio (min)',
    title='Tempo medio di completamento del sondaggio per tratto dominante',
    labels={'Tratto': '', 'Tempo medio (min)': 'Tempo medio (minuti)'},
    text='Tempo medio (min)'
)

fig.update_traces(texttemplate='%{text:.1f} min', textposition='outside')
fig.update_layout(yaxis_range=[0, tempo_per_tratto['Tempo medio (min)'].max() * 1.2])

fig.show()

## 6. Distribuzione dell'instabilità emotiva nel campione

L'instabilità emotiva è l'opposto della Stabilità emotiva: un punteggio alto
indica una persona più propensa ad ansia, stress e sbalzi d'umore.

Il grafico mostra come si distribuisce questo tratto nel campione, evidenziando
la soglia oltre la quale il livello di instabilità emotiva può essere considerato
elevato. Quante persone vivono con livelli estremi di stress?

In [7]:
# Calcolo l'instabilità emotiva come opposto della stabilità emotiva
df['Instabilità_emotiva'] = 6 - df['Stabilità_emotiva']

# Aggiungo una colonna che indica se la persona è sopra o sotto la soglia
df['Zona'] = df['Instabilità_emotiva'].apply(
    lambda x: 'Alto stress (oltre 4.0)' if x >= 4.0 else 'Nella norma'
)

# Istogramma
fig = px.histogram(
    df,
    x='Instabilità_emotiva',
    color='Zona',
    nbins=40,
    opacity=1,
    title='Distribuzione del neuroticismo',
    labels={
        'Instabilità_emotiva': 'Punteggio di neuroticismo (1–5)',
        'Zona': ''
    },
    color_discrete_map={
        'Alto stress (oltre 4.0)': 'red',
        'Nella norma': 'steelblue'
    }
)

# Opacità delle barre
fig.for_each_trace(lambda t: t.update(opacity=0.5) if t.name == 'Nella norma' else t.update(opacity=1))

# Linea verticale sulla soglia
fig.add_vline(
    x=4.0,
    line_dash='dash',
    line_color='darkred',
    annotation_text='Soglia (4.0)',
    annotation_position='top right'
)

fig.update_layout(
    yaxis_title='Numero di persone',
    xaxis_range=[1, 5],
    height=600
)

fig.show()

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/2183590888.py:2: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

/var/folders/w6/7cm71xss70n37td3zgkt_5xh0000gn/T/ipykernel_97451/2183590888.py:5: PerformanceWarning:

DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`



## 7. Correlazione tra i tratti di personalità

Analizziamo quanto i 5 tratti siano correlati tra loro — ovvero se tendono
ad andare insieme o meno.

Per esempio: una persona più estroversa tende ad essere anche più amichevole?
Chi è più coscienzioso tende ad essere anche più stabile emotivamente?

In [ ]:
# Calcolo la matrice di correlazione tra tutti i tratti
tratti = ['Estroversione', 'Stabilità_emotiva', 'Amabilità', 'Coscienziosità', 'Apertura_mentale']
matrice = df[tratti].corr().round(2)

# Heatmap
fig = go.Figure(data=go.Heatmap(
    z=matrice.values,          # i valori della matrice
    x=matrice.columns,         
    y=matrice.index,           
    colorscale='RdBu_r',         
    zmid=0,                    
    texttemplate='%{z:.2f}',   # mostra il valore numerico in ogni cella
    showscale=False
))

fig.update_layout(
    title='Correlazione tra i 5 tratti di personalità',
    xaxis_title='',
    yaxis_title=''
)

fig.show()

## 8. Punteggi medi per paese

Confrontiamo i punteggi medi dei 5 tratti di personalità tra i paesi
più rappresentati nel campione. L'obiettivo è capire se esistono differenze
culturali nei profili di personalità. Ad esempio, se alcune nazioni tendono
ad essere mediamente più estroverse o più coscienziose di altre.

In [12]:
top15 = df[df['country'] != 'NONE']['country'].value_counts().head(15).index.tolist()
df_paesi = df[df['country'].isin(top15)]

medie_paese = df_paesi.groupby('country')[tratti].mean().reset_index()

fig = go.Figure(data=go.Heatmap(
    z=medie_paese[tratti].values,
    x=tratti,
    y=medie_paese['country'],
    colorscale='Reds',
    showscale=False,
    texttemplate='%{z:.2f}',
    textfont=dict(color='black')
))

fig.update_layout(
    title='Punteggi medi dei 5 tratti per paese (top 15)',
    xaxis_title='',
    yaxis_title='',
    height=500
)

fig.show()

**Nota**: come si può notare non c'è un vero e proprio stacco tra i punteggi medi dei vari tratti nei diversi paesi.

## 9. Tempo medio di completamento del sondaggio per paese

Confrontiamo il tempo medio impiegato per completare il sondaggio tra i vari paesi.

In [9]:
# Filtro solo i paesi con almeno 1000 rispondenti
paesi_validi = df['country'].value_counts()
paesi_validi = paesi_validi[paesi_validi >= 1000].index.tolist()
df_paesi = df[df['country'].isin(paesi_validi)]

# Calcolo il tempo medio di completamento per paese in minuti
tempo_paese = df_paesi.groupby('country')['tempo_completamento_min'].mean().reset_index()
tempo_paese.columns = ['Paese', 'Tempo medio (min)']

# Ordine crescente
tempo_paese = tempo_paese.sort_values('Tempo medio (min)')

# Bar chart
fig = px.bar(
    tempo_paese,
    x='Paese',
    y='Tempo medio (min)',
    title='Tempo medio di completamento del sondaggio per paese (min. 1000 rispondenti)',
    labels={'Tempo medio (min)': 'Tempo medio (minuti)', 'Paese': ''},
)

fig.update_layout(
    yaxis_range=[0, tempo_paese['Tempo medio (min)'].max() * 1.2],
    xaxis_tickangle=45
)

fig.show()

In [10]:
tratti = ['Estroversione', 'Stabilità_emotiva', 'Amabilità', 'Coscienziosità', 'Apertura_mentale']

# Calcolo la correlazione tra ogni tratto e il tempo di completamento
correlazioni = df[tratti + ['tempo_completamento_min']].corr()['tempo_completamento_min'].drop('tempo_completamento_min')

df_corr = pd.DataFrame({
    'Tratto': correlazioni.index,
    'Correlazione': correlazioni.values
}).sort_values('Correlazione')

# Bar chart orizzontale
fig = px.bar(
    df_corr,
    x='Correlazione',
    y='Tratto',
    orientation='h',
    title='Correlazione tra i tratti di personalità e il tempo di completamento',
    labels={'Correlazione': 'Correlazione con il tempo di completamento', 'Tratto': ''},
    text='Correlazione',
    color='Correlazione',
    color_continuous_scale='RdBu',
)

fig.update_traces(texttemplate='%{text:.2f}', textposition='outside')
fig.update_layout(
    xaxis_range=[-1, 1],
    coloraxis_showscale=False
)

fig.show()

**Nota**: Non sembra esserci alcuna correlazione tra il tempo di completamento e i tratti della personalità.